<a href="https://colab.research.google.com/github/WallofWonder/Hugging-Face-RL-Learning/blob/main/notebooks/unit4/unit4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 单元 4：使用 PyTorch 从零手写你的第一个深度强化学习算法：Reinforce，并测试其鲁棒性 💪

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit6/thumbnail.png" alt="thumbnail"/>

在本笔记本中，你将从头开始编写你的第一个深度强化学习算法：Reinforce（也称为蒙特卡洛策略梯度，Monte Carlo Policy Gradient）。

Reinforce 是一种**基于策略的方法**（Policy-based method）：这是一种**尝试直接优化策略，而不使用动作价值函数**的深度强化学习算法。

更准确地说，Reinforce 属于**策略梯度方法**（Policy-gradient method），它是基于策略的方法的一个子类，旨在**通过使用梯度上升直接估计最优策略的权重来优化策略**。

为了测试其鲁棒性，我们将在 2 个不同的简单环境中对其进行训练：
- Cartpole-v1
- PixelcopterEnv

⬇️ 以下是**你在本笔记本结束时将实现的效果示例**。⬇️

  <img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit6/envs.gif" alt="Environments"/>

### 🎮 运行环境：

- [CartPole-v1](https://www.gymlibrary.dev/environments/classic_control/cart_pole/)
- [PixelCopter](https://pygame-learning-environment.readthedocs.io/en/latest/user/games/pixelcopter.html)

### 📚 强化学习库：

- Python
- PyTorch

我们一直在努力改进我们的教程，所以**如果你在本笔记本中发现任何问题**，请在 [GitHub 仓库上提交 Issue](https://github.com/huggingface/deep-rl-class/issues)。

## 本笔记本的目标 🏆
在学习完本笔记本后，你将能够：
- **使用 PyTorch 从零开始编写 Reinforce 算法。**
- **在简单环境里测试你的智能体的鲁棒性。**
- **将训练好的智能体推送至 Hugging Face Hub**，并附带精彩的视频回放和评估分数 🔥。

## 本笔记本来自《深度强化学习课程》
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/deep-rl-course-illustration.jpg" alt="Deep RL Course illustration"/>

在这个免费课程中，你将：

- 📖 系统地学习深度强化学习的**理论与实践**。
- 🧑‍💻 学习**使用著名的深度强化学习库**，如 Stable Baselines3、RL Baselines3 Zoo、CleanRL 和 Sample Factory 2.0。
- 🤖 在**独特的游戏环境中训练智能体**

更多信息请查看课程大纲 👉 https://simoninithomas.github.io/deep-rl-course

不要忘记**<a href="http://eepurl.com/ic5ZUD">订阅我们的课程邮件</a>**（我们收集您的邮箱，以便在每单元发布时给您发送链接，并提供关于挑战赛和内容更新的信息）。

保持联系的最佳方式是加入我们的 Discord 服务器，与社区及我们进行交流 👉🏻 https://discord.gg/ydHrjt3WP5

## 必备条件 🏗️
在开始本笔记本之前，你需要：

🔲 📚 [阅读第 4 单元来学习策略梯度理论](https://huggingface.co/deep-rl-course/unit4/introduction)

# 让我们开始从零手写 Reinforce 算法吧 🔥

为了通过这一动手实践的认证，你需要将训练好的模型上传到 Hub。

- 在 `Cartpole-v1` 获得 >= 350 的分数。
- 在 `PixelCopter` 获得 >= 5 的分数。

要想查看你的成绩，请访问排行榜并找到你的模型，**最终得分 = 平均奖励 - 奖励标准差**。**如果你在排行榜上没有看到你的模型，请滚动到页面底部并点击刷新按钮**。

关于证书申请流程的更多信息，请参阅此章节 👉 https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process

## 温馨提示 💡
建议在你的 Google 云端硬盘中保存一份此 Colab 的副本，这样**如果连接超时断开**，你仍然在云端硬盘中留有已保存的笔记本，不需要重新从头填写一切。

## 设置 GPU 运算 💪
- 为了**加速智能体的训练，我们将使用 GPU**。为此，请转至 `代码执行程序 > 更改运行时类型`。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step1.jpg" alt="GPU Step 1">

- 选择 `硬件加速器 > GPU`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step2.jpg" alt="GPU Step 2">

## 创建虚拟显示屏 🖥

在运行本笔记本的过程中，我们需要生成录像视频。在 Colab 环境中，**我们需要一个虚拟屏幕来渲染游戏环境**（从而记录每一帧画面）。

因此，以下单元格将安装所需库并启动虚拟屏幕 🖥

In [ ]:
%%capture
!apt install python-opengl
!apt install ffmpeg
!apt install xvfb
!pip install pyvirtualdisplay
!pip install pyglet==1.5.1

In [ ]:
# 虚拟显示器
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

## 安装依赖项 🔽
第一步是安装所需的库。我们将安装以下几项：

- `gym`
- `gym-games`：基于 PyGame 开发的额外 Gym 环境。
- `huggingface_hub`：🤗 作为一个中心平台，任何人都可以轻松分享和探索模型、数据集。它具有版本控制、指标评估和可视化等功能，能让你方便地与他人协作。

你可能会疑惑，为什么我们安装的是 gym 而不是更新的 gymnasium 呢？**因为我们使用的 gym-games 库目前尚未更新支持 gymnasium**。

在这里你将遇到的差异：
- 在 `gym` 中没有 `terminated` 和 `truncated`，只有唯一的 `done` 信号。
- 在 `gym` 中使用 `env.step()` 返回的是四个值：`state, reward, done, info`

你可以在这里了解关于 Gym 和 Gymnasium 的迁移差异 👉 https://gymnasium.farama.org/content/migration-guide/

你可以在这里查看所有可用的 Reinforce 模型 👉 https://huggingface.co/models?other=reinforce

你可以在这里找到所有的深度强化学习模型 👉 https://huggingface.co/models?pipeline_tag=reinforcement-learning

In [ ]:
!pip install -r https://raw.githubusercontent.com/huggingface/deep-rl-class/main/notebooks/unit4/requirements-unit4.txt

## 导入开发包 📦
除了导入已安装的库之外，我们还导入：

- `imageio`：一个帮助我们生成回放视频的库

In [ ]:
import numpy as np

from collections import deque

import matplotlib.pyplot as plt
%matplotlib inline

# PyTorch 框架
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

# Gym 环境
import gym
import gym_pygame

# Hugging Face Hub 工具
from huggingface_hub import notebook_login # 用于登录 Hugging Face 账号以将模型上传到 Hub。
import imageio

## 检查是否启用 GPU

- 让我们检查是否成功配置了 GPU
- 如果配置成功，你应该会看到输出 `device:cuda0`

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
print(device)

我们现在准备好手写我们的 Reinforce 算法了 🔥

# 第一个智能体：游玩 CartPole-v1 🤖

## 创建 CartPole 环境并理解其工作原理
### [游戏环境介绍 🎮](https://www.gymlibrary.dev/environments/classic_control/cart_pole/)

### 为什么我们使用像 CartPole-v1 这样简单的环境？
正如[强化学习技巧与窍门](https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html)中所解释的，当你从头实现智能体时，你需要**确保它在简单环境上能正常工作并找出 bug，然后再尝试更复杂的任务**。因为在简单的环境中定位 bug 会容易得多。


> 试着在玩具（简单）问题上看到“生命迹象”


> 通过在越来越难的环境中运行来验证实现（你可以将结果与 RL zoo 进行对比）。在这个步骤中，你通常需要运行超参数优化。
___
### CartPole-v1 环境

> 倒立摆连接在沿着无摩擦轨道移动的小车上。摆杆初始时呈直立状态，目标是通过对小车施加向左和向右的作用力来保持摆杆的平衡。

因此，我们从 CartPole-v1 开始。目标是向左或向右推动小车，**使摆杆保持在平衡状态**。

单局游戏（Episode）在以下情况结束：
- 摆杆角度大于 ±12°
- 小车位置大于 ±2.4
- 单局步数超过 500 步

只要摆杆保持平衡，我们每一步都会获得 +1 的奖励 💰。

In [ ]:
env_id = "CartPole-v1"
# 创建环境
env = gym.make(env_id)

# 创建用于评估的环境
eval_env = gym.make(env_id)

# 获取状态空间和动作空间的大小
s_size = env.observation_space.shape[0]
a_size = env.action_space.n

In [ ]:
print("_____观测空间_____ \n")
print("状态空间维度为: ", s_size)
print("样本观测数据", env.observation_space.sample()) # 获取一个随机状态样本

In [ ]:
print("\n _____动作空间_____ \n")
print("动作空间维度为: ", a_size)
print("动作空间样本", env.action_space.sample()) # 获取一个随机动作

## 构建 Reinforce 网络架构
本实现基于以下两个经典版本实现：
- [PyTorch 官方强化学习示例](https://github.com/pytorch/examples/blob/main/reinforcement_learning/reinforce.py)
- [Udacity Reinforce 实现](https://github.com/udacity/deep-reinforcement-learning/blob/master/reinforce/REINFORCE.ipynb)
- [由 Chris1nexus 改进的融合版本](https://github.com/huggingface/deep-rl-class/pull/95)

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit6/reinforce.png" alt="Reinforce"/>

我们期望：
- 两个全连接层（fc1 和 fc2）。
- 使用 ReLU 作为 fc1 的激活函数。
- 使用 Softmax 输出动作上的概率分布。

In [ ]:
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super(Policy, self).__init__()
        # 创建两个全连接层



    def forward(self, x):
        # 定义前向传播
        # 状态输入 fc1 然后应用 ReLU 激活函数

        # fc1 的输出输入 fc2

        # 最后输出 softmax

    def act(self, state):
        """
        给定一个状态，采取动作
        """
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = np.argmax(m)
        return action.item(), m.log_prob(action)

### 答案参考

In [ ]:
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super(Policy, self).__init__()
        self.fc1 = nn.Linear(s_size, h_size)
        self.fc2 = nn.Linear(h_size, a_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.softmax(x, dim=1)

    def act(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = np.argmax(m)
        return action.item(), m.log_prob(action)

我故意留下了一个错误，你能猜到在哪吗？

- 让我们运行一次前向传播来看一下：

In [ ]:
debug_policy = Policy(s_size, a_size, 64).to(device)
debug_policy.act(env.reset())

- 这里我们看到报错 `ValueError: The value argument to log_prob must be a Tensor`

- 这意味着 `m.log_prob(action)` 中的 `action` 必须是一个张量（Tensor），**但它现在不是。**

- 你知道为什么吗？检查一下 `act` 函数，试着找出无法运行的原因。

提示 💡：此处的实现有一些问题。记住在 `act` 函数中**我们希望从动作的概率分布中进行采样**。

### （真正的）解决方案

In [ ]:
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super(Policy, self).__init__()
        self.fc1 = nn.Linear(s_size, h_size)
        self.fc2 = nn.Linear(h_size, a_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.softmax(x, dim=1)

    def act(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action)

通过使用 CartPole，调试变得更加简单，因为**我们确信 bug 来自于我们的代码编写，而不是由于复杂的环境导致的**。

- 既然**我们希望从动作概率分布中进行采样**，我们就不能使用 `action = np.argmax(m)`，因为它总是只输出概率最高的那个动作。

- 我们需要替换为 `action = m.sample()`，它会根据概率分布 P(.|s) 随机采样一个动作。

### 构建 Reinforce 训练算法
这是 Reinforce 算法的伪代码：

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit6/pg_pseudocode.png" alt="Policy gradient pseudocode"/>

- 当我们计算回报 $G_t$（第 6 行）时，我们发现我们是在计算**从时间步 t 开始**的折扣奖励总和。

- 为什么？因为我们的策略应该**只依据后续的后果来强化动作**：所以在采取某个动作之前获得的奖励是无用的（因为它们不是由当前动作引起的），**只有动作发生之后的奖励才起作用**。

- 在编写代码之前，你应该阅读 [don't let the past distract you](https://spinningup.openai.com/en/latest/spinningup/rl_intro3.html#don-t-let-the-past-distract-you) 这一章节，它解释了为什么我们要使用只针对未来的回报（reward-to-go）策略梯度。

我们使用由 [Chris1nexus](https://github.com/Chris1nexus) 编写的非常巧妙的技巧来**高效地计算每个时间步的回报**。注释解释了具体步骤，你也可以[查阅 PR 详细说明](https://github.com/huggingface/deep-rl-class/pull/95)。
但总的思路是**高效地在每个时间步计算回报**。

你可能会问的第二个问题是：**为什么我们要最小化损失函数（minimize the loss）**？你之前不是说要用梯度上升（Gradient Ascent）而不是梯度下降吗？

- 我们的目标是最大化效用函数 $J(\theta)$，但在 PyTorch 和 Tensorflow 等框架中，更习惯于**最小化一个目标函数（Loss）**。
    - 假设我们想在某个时间步强化动作 3。训练前该动作的概率 P 是 0.25。
    - 我们希望修改 $\theta$ 使得 $\pi_\theta(a_3|s; \theta) > 0.25$。
    - 因为所有概率之和必须为 1，最大化 $\pi_\theta(a_3|s; \theta)$ 就会自动**最小化其他动作的概率**。
    - 我们可以让 PyTorch 去**最小化 $1 - \pi_\theta(a_3|s; \theta)$**。
    - 当 $\pi_\theta(a_3|s; \theta)$ 接近 1 时，此损失函数接近 0。
    - 因此，我们通过这个最小化损失的过程来实现最大化 $\pi_\theta(a_3|s; \theta)$ 的梯度优化。

In [ ]:
def reinforce(policy, optimizer, n_training_episodes, max_t, gamma, print_every):
    # 用于在训练过程中计算滑动平均分数
    scores_deque = deque(maxlen=100)
    scores = []
    # 对应伪代码第 3 行
    for i_episode in range(1, n_training_episodes+1):
        saved_log_probs = []
        rewards = []
        state = # TODO: 重置环境
        # 对应伪代码第 4 行
        for t in range(max_t):
            action, log_prob = # TODO: 获取动作
            saved_log_probs.append(log_prob)
            state, reward, done, _ = # TODO: 在环境中执行一步
            rewards.append(reward)
            if done:
                break
        scores_deque.append(sum(rewards))
        scores.append(sum(rewards))

        # 对应伪代码第 6 行：计算回报
        returns = deque(maxlen=max_t)
        n_steps = len(rewards)
        # 计算每个时间步的折扣回报，
        # 即：时间步 t 的 gamma 折扣回报 (G_t) + 时间步 t 的即时奖励之和

        # 时间复杂度为 O(N)，其中 N 是总时间步数
        # （这个折扣回报 G_t 的定义遵循 Sutton&Barto 2017 第二版草案第 44 页的定义）
        # G_t = r_(t+1) + r_(t+2) + ...

        # 在此定义下，每个时间步 t 的回报可以重用已计算的未来回报 G_(t+1) 来进行计算：
        # G_t = r_(t+1) + gamma*G_(t+1)
        # G_(t-1) = r_t + gamma* G_t
        # （这遵循动态规划方法，通过记忆化中间结果来避免重复计算）

        # 这样做是正确的，因为这等价于（参见 Sutton&Barto 2017 第二版草案第 46 页）：
        # G_(t-1) = r_t + gamma*r_(t+1) + gamma*gamma*r_(t+2) + ...


        ## 综上，我们计算时间步 t 的回报公式为：
        #               gamma[t] * return[t] + reward[t]
        #
        ## 我们从最后一个时间步倒序计算到第一个时间步，以便应用上述递推公式
        ## 并避免从前到后计算时所产生的冗余计算。

        ## 因此，“returns” 双端队列将按时间顺序保存从 t=0 到 t=n_steps 的回报，
        ## 借助 appendleft() 函数，我们可以以 O(1) 的常数时间将其插入到队首，
        ## 而普通的 Python 列表则需要 O(N) 的时间。
        for t in range(n_steps)[::-1]:
            disc_return_t = (returns[0] if len(returns)>0 else 0)
            returns.appendleft(    ) # TODO: 在这里完成代码

        ## 对回报进行归一化（标准化）处理，以使训练更加稳定
        eps = np.finfo(np.float32).eps.item()

        ## eps 是单精度浮点数能表示的最小值，
        # 将其加到回报的标准差中以避免发生除零等数值不稳定问题
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + eps)

        # 对应第 7 行：
        policy_loss = []
        for log_prob, disc_return in zip(saved_log_probs, returns):
            policy_loss.append(-log_prob * disc_return)
        policy_loss = torch.cat(policy_loss).sum()

        # 对应第 8 行：PyTorch 默认进行梯度下降，所以加负号
        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        if i_episode % print_every == 0:
            print('Episode {}\tAverage Score: {:.2f}'.format(i_episode, np.mean(scores_deque)))

    return scores

#### 答案参考

In [ ]:
def reinforce(policy, optimizer, n_training_episodes, max_t, gamma, print_every):
    # 用于在训练过程中计算滑动平均分数
    scores_deque = deque(maxlen=100)
    scores = []
    # 对应伪代码第 3 行
    for i_episode in range(1, n_training_episodes+1):
        saved_log_probs = []
        rewards = []
        state = env.reset()
        # 对应伪代码第 4 行
        for t in range(max_t):
            action, log_prob = policy.act(state)
            saved_log_probs.append(log_prob)
            state, reward, done, _ = env.step(action)
            rewards.append(reward)
            if done:
                break
        scores_deque.append(sum(rewards))
        scores.append(sum(rewards))

        # 对应伪代码第 6 行：计算回报
        returns = deque(maxlen=max_t)
        n_steps = len(rewards)
        # 计算每个时间步的折扣回报，
        # 即：
        #      时间步 t 的 gamma 折扣回报 (G_t) + 时间步 t 的即时奖励之和
        #
        # 时间复杂度为 O(N)，其中 N 是总时间步数
        # （这个折扣回报 G_t 的定义遵循 Sutton&Barto 2017 第二版草案第 44 页的定义）
        # G_t = r_(t+1) + r_(t+2) + ...

        # 在此定义下，每个时间步 t 的回报可以重用已计算的未来回报 G_(t+1) 来进行计算：
        # G_t = r_(t+1) + gamma*G_(t+1)
        # G_(t-1) = r_t + gamma* G_t
        # （这遵循动态规划方法，通过记忆化中间结果来避免重复计算）

        # 这样做是正确的，因为这等价于（参见 Sutton&Barto 2017 第二版草案第 46 页）：
        # G_(t-1) = r_t + gamma*r_(t+1) + gamma*gamma*r_(t+2) + ...


        ## 综上，我们计算时间步 t 的回报公式为：
        #               gamma[t] * return[t] + reward[t]
        #
        ## 我们从最后一个时间步倒序计算到第一个时间步，以便应用上述递推公式
        ## 并避免从前到后计算时所产生的冗余计算。

        ## 因此，“returns” 双端队列将按时间顺序保存从 t=0 到 t=n_steps 的回报，
        ## 借助 appendleft() 函数，我们可以以 O(1) 的常数时间将其插入到队首，
        ## 而普通的 Python 列表则需要 O(N) 的时间。
        for t in range(n_steps)[::-1]:
            disc_return_t = (returns[0] if len(returns)>0 else 0)
            returns.appendleft( gamma*disc_return_t + rewards[t]   )

        ## 对回报进行归一化（标准化）处理，以使训练更加稳定
        eps = np.finfo(np.float32).eps.item()
        ## eps 是单精度浮点数能表示的最小值，
        # 将其加到回报的标准差中以避免发生除零等数值不稳定问题
        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + eps)

        # 对应第 7 行：
        policy_loss = []
        for log_prob, disc_return in zip(saved_log_probs, returns):
            policy_loss.append(-log_prob * disc_return)
        policy_loss = torch.cat(policy_loss).sum()

        # 对应第 8 行：PyTorch 默认进行梯度下降，所以加负号
        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        if i_episode % print_every == 0:
            print('Episode {}\tAverage Score: {:.2f}'.format(i_episode, np.mean(scores_deque)))

    return scores

## 开始训练
- 我们现在准备好训练我们的智能体了。
- 首先，我们定义一个包含所有训练超参数的字典。
- 你可以随时修改并微调这些训练超参数 😉

In [ ]:
cartpole_hyperparameters = {
    "h_size": 16,
    "n_training_episodes": 1000,
    "n_evaluation_episodes": 10,
    "max_t": 1000,
    "gamma": 1.0,
    "lr": 1e-2,
    "env_id": env_id,
    "state_space": s_size,
    "action_space": a_size,
}

In [ ]:
# 创建策略网络并将其移至指定的计算设备（如 GPU）
cartpole_policy = Policy(cartpole_hyperparameters["state_space"], cartpole_hyperparameters["action_space"], cartpole_hyperparameters["h_size"]).to(device)
cartpole_optimizer = optim.Adam(cartpole_policy.parameters(), lr=cartpole_hyperparameters["lr"])

In [ ]:
scores = reinforce(cartpole_policy,
                   cartpole_optimizer,
                   cartpole_hyperparameters["n_training_episodes"],
                   cartpole_hyperparameters["max_t"],
                   cartpole_hyperparameters["gamma"],
                   100)

## 定义评估方法 📝
- 这里我们定义了用于测试 Reinforce 智能体性能的评估方法。

In [ ]:
def evaluate_agent(env, max_steps, n_eval_episodes, policy):
  """
  对智能体进行 ``n_eval_episodes`` 轮评估，并返回平均奖励和奖励标准差。
  :param env: 评估所用的环境
  :param n_eval_episodes: 评估智能体的局数
  :param policy: 待评估的 Reinforce 智能体
  """
  episode_rewards = []
  for episode in range(n_eval_episodes):
    state = env.reset()
    step = 0
    done = False
    total_rewards_ep = 0

    for step in range(max_steps):
      action, _ = policy.act(state)
      new_state, reward, done, info = env.step(action)
      total_rewards_ep += reward

      if done:
        break
      state = new_state
    episode_rewards.append(total_rewards_ep)
  mean_reward = np.mean(episode_rewards)
  std_reward = np.std(episode_rewards)

  return mean_reward, std_reward

## 评估我们的智能体 📈

In [ ]:
evaluate_agent(eval_env,
               cartpole_hyperparameters["max_t"],
               cartpole_hyperparameters["n_evaluation_episodes"],
               cartpole_policy)

### 在 Hugging Face Hub 上发布我们训练好的模型 🔥
现在我们看到训练后取得了不错的结果，我们可以只用一行代码将我们训练好的模型发布到 Hub 🤗。

下面是模型卡片（Model Card）的展示示例：

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit6/modelcard.png"/>

### 推送至 Hub
#### 请不要修改以下代码

In [ ]:
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.repocard import metadata_eval_result, metadata_save

from pathlib import Path
import datetime
import json
import imageio

import tempfile

import os

In [ ]:
def record_video(env, policy, out_directory, fps=30):
  """
  生成智能体游玩的录像视频
  :param env: 环境
  :param policy: 我们的智能体策略模型
  :param out_directory: 视频输出路径
  :param fps: 每秒帧数（对于 taxi-v3 和 frozenlake-v1 我们使用 1）
  """
  images = []
  done = False
  state = env.reset()
  img = env.render(mode='rgb_array')
  images.append(img)
  while not done:
    # 采取当前状态下输出概率最大的动作
    action, _ = policy.act(state)
    state, reward, done, info = env.step(action) # 更新状态以便进行下一步记录
    img = env.render(mode='rgb_array')
    images.append(img)
  imageio.mimsave(out_directory, [np.array(img) for i, img in enumerate(images)], fps=fps)

In [ ]:
def push_to_hub(repo_id,
                model,
                hyperparameters,
                eval_env,
                video_fps=30
                ):
  """
  评估模型，录制视频，并将模型上传到 Hugging Face Hub。
  该方法执行完整的流水线：
  - 评估模型性能
  - 自动生成模型卡片（README）
  - 录制智能体玩游戏的演示视频
  - 将所有内容一键推送到 Hub

  :param repo_id: Hugging Face Hub 上的模型仓库 ID
  :param model: 我们想要保存的 PyTorch 模型
  :param hyperparameters: 训练时的超参数
  :param eval_env: 评估所用的环境
  :param video_fps: 录制游戏回放视频的帧率
  """

  _, repo_name = repo_id.split("/")
  api = HfApi()

  # 步骤 1：创建仓库
  repo_url = api.create_repo(
        repo_id=repo_id,
        exist_ok=True,
  )

  with tempfile.TemporaryDirectory() as tmpdirname:
    local_directory = Path(tmpdirname)

    # 步骤 2：保存 PyTorch 模型
    torch.save(model, local_directory / "model.pt")

    # 步骤 3：将超参数保存为 JSON 文件
    with open(local_directory / "hyperparameters.json", "w") as outfile:
      json.dump(hyperparameters, outfile)

    # 步骤 4：评估模型性能并构建结果 JSON
    mean_reward, std_reward = evaluate_agent(eval_env,
                                            hyperparameters["max_t"],
                                            hyperparameters["n_evaluation_episodes"],
                                            model)
    # 获取当前日期时间
    eval_datetime = datetime.datetime.now()
    eval_form_datetime = eval_datetime.isoformat()

    evaluate_data = {
          "env_id": hyperparameters["env_id"],
          "mean_reward": mean_reward,
          "n_evaluation_episodes": hyperparameters["n_evaluation_episodes"],
          "eval_datetime": eval_form_datetime,
    }

    # 写入 JSON 结果文件
    with open(local_directory / "results.json", "w") as outfile:
        json.dump(evaluate_data, outfile)

    # 步骤 5：构建模型卡片元数据
    env_name = hyperparameters["env_id"]

    metadata = {}
    metadata["tags"] = [
          env_name,
          "reinforce",
          "reinforcement-learning",
          "custom-implementation",
          "deep-rl-class"
      ]

    # 添加评估指标
    eval = metadata_eval_result(
        model_pretty_name=repo_name,
        task_pretty_name="reinforcement-learning",
        task_id="reinforcement-learning",
        metrics_pretty_name="mean_reward",
        metrics_id="mean_reward",
        metrics_value=f"{mean_reward:.2f} +/- {std_reward:.2f}",
        dataset_pretty_name=env_name,
        dataset_id=env_name,
      )

    # 合并元数据字典
    metadata = {**metadata, **eval}

    model_card = f"""
  # **Reinforce** Agent playing **{env_id}**
  This is a trained model of a **Reinforce** agent playing **{env_id}** .
  To learn to use this model and train yours check Unit 4 of the Deep Reinforcement Learning Course: https://huggingface.co/deep-rl-course/unit4/introduction
  """

    readme_path = local_directory / "README.md"
    readme = ""
    if readme_path.exists():
        with readme_path.open("r", encoding="utf8") as f:
          readme = f.read()
    else:
      readme = model_card

    with readme_path.open("w", encoding="utf-8") as f:
      f.write(readme)

    # 将指标保存到 Readme 的元数据区域中
    metadata_save(readme_path, metadata)

    # 步骤 6：录制回放视频
    video_path =  local_directory / "replay.mp4"
    record_video(env, model, video_path, video_fps)

    # 步骤 7：将所有文件一并上传至 Hub
    api.upload_folder(
          repo_id=repo_id,
          folder_path=local_directory,
          path_in_repo=".",
    )

    print(f"您的模型已成功推送到 Hub。您可以在这里查看您的模型: {repo_url}")

### .

通过使用 `push_to_hub` **你将对模型进行评估、录制回放视频、生成模型卡片并一键推送至 Hugging Face Hub**。

通过这种方式：
- 你可以**展示你的丰硕成果** 🔥
- 你可以**在线观看你的智能体玩游戏的过程** 👀
- 你可以**与社区分享你的模型，供其他人直接使用** 💾
- 你可以**访问排行榜 🏆 看看你的智能体与其他同学相比表现如何** 👉 https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard

要想将你的模型分享给社区，只需遵循以下三个步骤：

1️⃣ （如果尚未注册）注册一个 HF 账号 ➡ https://huggingface.co/join

2️⃣ 登录并获取你账号的认证 Token。
- 创建一个新 Token (https://huggingface.co/settings/tokens)，**请务必赋予其 "write"（写入）权限**。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/create-token.jpg" alt="Create HF Token">

In [ ]:
notebook_login()

如果你不想在 Google Colab 或 Jupyter Notebook 中使用交互登录，你可以在终端或环境中使用此命令来登录：`huggingface-cli login`（或直接 `login`）

3️⃣ 我们现在可以使用 `push_to_hub()` 函数将我们训练好的智能体推送到 🤗 Hub 啦 🔥

In [ ]:
repo_id = "" #TODO 定义您的仓库 ID {用户名/Reinforce-{model-id}}
push_to_hub(repo_id,
                cartpole_policy, # 我们想要保存的模型
                cartpole_hyperparameters, # 超参数
                eval_env, # 评估环境
                video_fps=30
                )

现在我们已经验证了实现的鲁棒性，让我们来挑战一个更复杂的环境：PixelCopter 🚁

## 第二个智能体：PixelCopter 🚁

### 观察并熟悉 PixelCopter 环境 👀
- [游戏环境文档](https://pygame-learning-environment.readthedocs.io/en/latest/user/games/pixelcopter.html)

In [ ]:
env_id = "Pixelcopter-PLE-v0"
env = gym.make(env_id)
eval_env = gym.make(env_id)
s_size = env.observation_space.shape[0]
a_size = env.action_space.n

In [ ]:
print("_____OBSERVATION SPACE_____ \n")
print("The State Space is: ", s_size)
print("Sample observation", env.observation_space.sample()) # Get a random observation

In [ ]:
print("\n _____ACTION SPACE_____ \n")
print("The Action Space is: ", a_size)
print("Action Space Sample", env.action_space.sample()) # Take a random action

观测空间包含 7 个特征 👀：
- 玩家的 Y 坐标位置
- 玩家的速度
- 玩家与地面的距离
- 玩家与天花板的距离
- 下一个障碍物与玩家的 X 轴水平距离
- 下一个障碍物顶部的 Y 轴位置
- 下一个障碍物底部的 Y 轴位置

动作空间包含 2 个动作 🎮：
- 向上（按下加速按钮）
- 什么都不做（不按加速按钮）

奖励函数设计 💰：
- 智能体每穿过一个垂直障碍物，就会获得一个 +1 的正向奖励。每一次进入终止状态（坠毁），将获得 -1 的惩罚奖励。

### 定义新的策略网络 🧠
- 既然环境变得更加复杂，我们需要一个更深层的神经网络

In [ ]:
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super(Policy, self).__init__()
        # 在这里定义三个网络层

    def forward(self, x):
        # 在这里定义前向传播计算过程
        return F.softmax(x, dim=1)

    def act(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action)

### 答案参考

In [ ]:
class Policy(nn.Module):
    def __init__(self, s_size, a_size, h_size):
        super(Policy, self).__init__()
        self.fc1 = nn.Linear(s_size, h_size)
        self.fc2 = nn.Linear(h_size, h_size*2)
        self.fc3 = nn.Linear(h_size*2, a_size)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return F.softmax(x, dim=1)

    def act(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action)

### 定义超参数 ⚙️
- 鉴于这个环境更加复杂。
- 特别是隐藏层的大小（h_size），我们需要配置更多的神经元。

In [ ]:
pixelcopter_hyperparameters = {
    "h_size": 64,
    "n_training_episodes": 50000,
    "n_evaluation_episodes": 10,
    "max_t": 10000,
    "gamma": 0.99,
    "lr": 1e-4,
    "env_id": env_id,
    "state_space": s_size,
    "action_space": a_size,
}

### 开始训练
- 我们现在可以开始训练我们的智能体了 🔥。

In [ ]:
# 创建策略网络并将其放置在运算设备上
# torch.manual_seed(50)
pixelcopter_policy = Policy(pixelcopter_hyperparameters["state_space"], pixelcopter_hyperparameters["action_space"], pixelcopter_hyperparameters["h_size"]).to(device)
pixelcopter_optimizer = optim.Adam(pixelcopter_policy.parameters(), lr=pixelcopter_hyperparameters["lr"])

In [ ]:
scores = reinforce(pixelcopter_policy,
                   pixelcopter_optimizer,
                   pixelcopter_hyperparameters["n_training_episodes"],
                   pixelcopter_hyperparameters["max_t"],
                   pixelcopter_hyperparameters["gamma"],
                   1000)

### 将我们训练好的模型发布到 Hub 🔥

In [ ]:
repo_id = "" #TODO 定义您的仓库 ID {用户名/Reinforce-{model-id}}
push_to_hub(repo_id,
                pixelcopter_policy, # 我们想要保存的模型
                pixelcopter_hyperparameters, # 超参数
                eval_env, # 评估环境
                video_fps=30
                )

## 进阶额外挑战 🏆
学好强化学习最好的方法**就是自己亲自动手尝试不同的点子**！如你所见，目前的智能体表现还不够理想。作为最直接的建议，你可以尝试训练更多步数，也可以努力去搜寻更优的参数。

你可以在[官方排行榜](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard)上找到你上传的智能体。你能打榜到第一名吗？

这里有一些能够帮你提升效果的思路：
* 训练更多的步数/回合
* 参考其他同学优秀的参数来尝试不同的超参数组合 👉 https://huggingface.co/models?other=reinforce
* **重新上传你优化训练后的新模型** 到 Hub 🔥
* **针对更加复杂的环境改进算法实现**（例如，如果希望直接处理游戏画面帧作为输入，是否可以考虑将全连接层网络替换为卷积神经网络 CNN 呢？）

________________________________________________________________________

**恭喜你完成了本单元的学习**！这一章包含了非常多的硬核强化学习干货。
也非常高兴你能顺利走完这个实战。你刚刚使用 PyTorch 从头手写了你的第一个深度强化学习算法，并在 Hub 上与大家共同分享了你的成果 🥳。

不要犹豫，继续通过**针对更复杂的环境改进代码**来深入钻研（例如，将网络修改为卷积神经网络以利用原始图像作为观测输入）。

在接下来的单元中，**我们将深入探讨 Unity MLAgents**，在逼真的 Unity 仿真环境里训练智能体。通过这种方式，你将整装待发去参加 **AI vs AI 挑战赛**，在雪仗和足球比赛中训练你的智能体与其他同学的智能体一决高下。

听起来是不是很有趣？我们下节课再见！

最后，我们非常希望能**倾听你对本课程的看法和建议**。如果你有任何反馈，请花一分钟 👉 [填写此表单](https://forms.gle/BzKXWzLAGZESGNaE9)

第 5 单元见！保持热爱，继续前行！🔥

### 保持探索，你最棒 🤗